# A0 Full-Test Checkpoint Action Distribution

This notebook builds action-distribution plots from the checkpoint recorded in `outputs/full_test_eval/*.json`. It does **not** use the old mean of the last 5 logged W&B points. The full-test JSON selects the checkpoint and step; cached history supplies the nearest requested evaluation action metric at that step.

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import action_distribution_metrics as adm

print("Task dir:", TASK_DIR)
print("Helper dir:", HELPER_DIR)

In [ ]:
# Choose which cached run-data source to use for histories.
# Use "cpu", "gpu", or "all". Unsplit folders such as a0_hvg are kept as shared baselines.
RUN_DATA_SOURCE = "all"

# Keep this broad to include future a0_sparse16/a0_aib full-test JSONs when they exist.
FULL_TEST_CHECKPOINT_REGEX = r"^a0_"

# W&B histories do not always have an eval row exactly at checkpoint_global_step.
# "at_or_before" uses the latest eval metric at or before the checkpoint step and reports the delta.
# Other accepted values: "exact", "nearest".
STEP_MATCH_POLICY = "at_or_before"
MAX_STEP_DELTA = None  # set an integer number of env steps to reject distant cached eval points

CONFIG_FOLDERS_TO_LOAD = ["a0_hvg", "a0_sparse16", "a0_aib"]
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"
SAVE_FIGURES = True
SHOW_FIGURES = True

# Keep broad debug tables hidden by default.
SHOW_DIAGNOSTIC_TABLES = False

# Show the exact checkpoint files used before each checkpoint-aligned plot.
SHOW_CHECKPOINT_AUDIT_TABLES = True

## Full-Test / Checkpoint Coverage

In [ ]:
full_test = adm.load_full_test_eval_results()
pattern = re.compile(FULL_TEST_CHECKPOINT_REGEX)
a0_full_test = full_test[
    full_test["run_like"].fillna("").astype(str).map(lambda value: bool(pattern.search(value)))
].copy()

if SHOW_DIAGNOSTIC_TABLES:
    print(f"A0 full-test eval JSON rows matching {FULL_TEST_CHECKPOINT_REGEX!r}: {len(a0_full_test)}")
    adm.print_full_test_eval_source_folders(a0_full_test)
    display(a0_full_test[[
        "run_like",
        "checkpoint_global_step",
        "survival_percent",
        "split",
        "eval_episodes",
        "local_checkpoint_exists",
        "local_checkpoint_path",
        "local_best_test_checkpoint_path",
        "path",
    ]].sort_values(["run_like", "checkpoint_global_step"]))


In [ ]:
checkpoint_dir = TASK_DIR / "checkpoint" / "with_obs_stats"
local_best = pd.DataFrame(
    {
        "checkpoint_path": [str(path) for path in sorted(checkpoint_dir.glob("best_test_a0_*.tar"))]
    }
)
if not local_best.empty:
    local_best["run_like"] = local_best["checkpoint_path"].map(lambda value: Path(value).stem.removeprefix("best_test_"))
    full_test_run_likes = set(a0_full_test["run_like"].dropna().astype(str))
    missing_full_test = local_best[~local_best["run_like"].isin(full_test_run_likes)].copy()
else:
    missing_full_test = local_best

if SHOW_DIAGNOSTIC_TABLES:
    print(f"Local best_test_a0 checkpoints without matching full_test_eval JSON: {len(missing_full_test)}")
    display(missing_full_test)


## Load Cached Histories

In [ ]:
ctx = adm.load_action_distribution_context(
    experiment_folders=CONFIG_FOLDERS_TO_LOAD,
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    run_data_source=RUN_DATA_SOURCE,
    save_figures=SAVE_FIGURES,
    show_figures=SHOW_FIGURES,
)

if SHOW_DIAGNOSTIC_TABLES:
    display(ctx["coverage"].groupby(["experiment", "family_label"], dropna=False).agg(
        expected=("expected_run_name", "count"),
        cached=("cached", "sum"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
    ).reset_index())


## A0 HVG Baseline Action-0 By Seed

This plot reads `best_test_a0_hvg_00_baseline_s*.tar`, extracts each checkpoint `global_step`, and shows the cached evaluation action-0 / do-nothing fraction at that checkpoint step for every agent and seed.


In [ ]:
import plotly.express as px
import torch

BASELINE_RUN_REGEX = r"^a0_hvg_00_baseline_s[0-2]$"
BASELINE_AGENTS = ["agent_0", "agent_1", "agent_2"]
BEST_TEST_CHECKPOINT_DIR = TASK_DIR / "checkpoint" / "with_obs_stats"


def _seed_from_run_name(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else None


def _load_checkpoint_global_step(path):
    record = torch.load(path, map_location="cpu", weights_only=False)
    return int(record.get("global_step"))


def _checkpoint_audit_table(seed_rows, *, labels=None, plot_group=None):
    base_columns = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
        "checkpoint_path",
    ]
    if seed_rows is None or seed_rows.empty:
        return pd.DataFrame(columns=base_columns)

    audit = seed_rows.copy()
    if labels is not None and "condition_label" in audit.columns:
        audit = audit[audit["condition_label"].astype(str).isin([str(label) for label in labels])].copy()
    if plot_group is not None and "plot_group" in audit.columns:
        baseline_mask = audit.get("condition_label", pd.Series(index=audit.index, dtype=object)).astype(str).eq("baseline")
        audit = audit[audit["plot_group"].astype(str).eq(str(plot_group)) | baseline_mask].copy()

    if audit.empty:
        return pd.DataFrame(columns=base_columns)

    if "checkpoint_path" not in audit.columns:
        audit["checkpoint_path"] = ""
    audit["checkpoint_path"] = audit["checkpoint_path"].fillna("").astype(str)
    audit["checkpoint_file"] = audit["checkpoint_path"].map(lambda value: Path(value).name if value else "")

    optional_defaults = {
        "condition_label": "",
        "run_name": "",
        "seed": pd.NA,
        "checkpoint_global_step": pd.NA,
        "selected_metric_step": pd.NA,
        "step_delta": pd.NA,
        "step_match_policy": "",
    }
    for column, default in optional_defaults.items():
        if column not in audit.columns:
            audit[column] = default

    key_columns = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
        "checkpoint_path",
    ]
    result = audit[key_columns].drop_duplicates().copy()

    if {"agent", "action0_fraction"}.issubset(audit.columns):
        seed_value_table = (
            audit
            .pivot_table(
                index=key_columns,
                columns="agent",
                values="action0_fraction",
                aggfunc="first",
                observed=True,
            )
            .reset_index()
        )
        seed_value_table = seed_value_table.rename(
            columns={
                agent: f"{agent}_plotted_action0_fraction"
                for agent in seed_value_table.columns
                if str(agent).startswith("agent_")
            }
        )
        result = result.merge(seed_value_table, on=key_columns, how="left")

        mean_index_columns = ["condition_label"]
        if "plot_group" in audit.columns:
            mean_index_columns.append("plot_group")
        mean_value_table = (
            audit
            .pivot_table(
                index=mean_index_columns,
                columns="agent",
                values="action0_fraction",
                aggfunc="mean",
                observed=True,
            )
            .reset_index()
        )
        mean_value_table = mean_value_table.rename(
            columns={
                agent: f"{agent}_mean_plotted_action0_fraction"
                for agent in mean_value_table.columns
                if str(agent).startswith("agent_")
            }
        )
        join_columns = [column for column in mean_index_columns if column in result.columns]
        if join_columns:
            if "plot_group" not in result.columns and "plot_group" in mean_value_table.columns:
                mean_value_table = mean_value_table.drop(columns=["plot_group"])
            result = result.merge(mean_value_table, on=join_columns, how="left")

    sort_columns = [column for column in ["condition_label", "run_name", "seed", "checkpoint_file"] if column in result.columns]
    return result.sort_values(sort_columns, kind="stable").reset_index(drop=True)


def _display_checkpoint_audit(seed_rows, *, title, labels=None, plot_group=None):
    audit = _checkpoint_audit_table(seed_rows, labels=labels, plot_group=plot_group)
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print(f"Checkpoint files used for: {title}")
        display(audit)
    return audit


history = ctx["history_wide"].copy()
selected_runs = ctx["selected_runs"].copy()
baseline_runs = selected_runs[
    selected_runs["run_name"].fillna("").astype(str).str.match(BASELINE_RUN_REGEX)
].sort_values("run_name")

baseline_action0_rows = []
baseline_missing_rows = []
for run in baseline_runs.to_dict("records"):
    run_name = str(run["run_name"])
    run_id = str(run["run_id"])
    seed = _seed_from_run_name(run_name)
    checkpoint_path = BEST_TEST_CHECKPOINT_DIR / f"best_test_{run_name}.tar"
    if not checkpoint_path.exists():
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": "missing_best_test_checkpoint",
            "checkpoint_path": str(checkpoint_path),
        })
        continue

    try:
        checkpoint_step = _load_checkpoint_global_step(checkpoint_path)
    except Exception as exc:
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": f"checkpoint_load_failed: {type(exc).__name__}: {exc}",
            "checkpoint_path": str(checkpoint_path),
        })
        continue

    run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
    if run_history.empty:
        baseline_missing_rows.append({
            "run_name": run_name,
            "seed": seed,
            "reason": "missing_cached_history",
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_global_step": checkpoint_step,
        })
        continue

    for agent in BASELINE_AGENTS:
        values, metric_column, inverted = adm._action0_series_for_checkpoint_agent(
            run_history,
            str(EVAL_METRIC_SPLIT),
            agent,
        )
        if values is None:
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "missing_agent_action0_metric",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
            })
            continue

        selected_point = adm._select_checkpoint_metric_point(
            run_history,
            values,
            checkpoint_step,
            step_policy=STEP_MATCH_POLICY,
        )
        if selected_point is None:
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "no_metric_point_at_policy_step",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
                "metric_column": metric_column,
            })
            continue

        point, match_policy = selected_point
        selected_step = float(point["_step"])
        step_delta = selected_step - float(checkpoint_step)
        if MAX_STEP_DELTA is not None and abs(step_delta) > float(MAX_STEP_DELTA):
            baseline_missing_rows.append({
                "run_name": run_name,
                "seed": seed,
                "agent": agent,
                "reason": "step_delta_exceeds_max",
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_global_step": checkpoint_step,
                "selected_metric_step": selected_step,
                "step_delta": step_delta,
                "metric_column": metric_column,
            })
            continue

        baseline_action0_rows.append({
            "run_name": run_name,
            "run_id": run_id,
            "seed": seed,
            "seed_label": f"seed {seed}",
            "agent": agent,
            "action0_fraction": float(point["value"]),
            "metric_column": metric_column,
            "metric_inverted_from_nonidle": bool(inverted),
            "checkpoint_global_step": checkpoint_step,
            "checkpoint_step_millions": checkpoint_step / 1_000_000,
            "selected_metric_step": selected_step,
            "selected_metric_step_millions": selected_step / 1_000_000,
            "step_delta": step_delta,
            "step_match_policy": match_policy,
            "checkpoint_path": str(checkpoint_path),
        })

baseline_action0_by_seed = pd.DataFrame(baseline_action0_rows).sort_values(["agent", "seed"])
baseline_action0_missing = pd.DataFrame(baseline_missing_rows)

print(f"Best-test checkpoint-aligned action-0 rows for a0_hvg_00_baseline: {len(baseline_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(baseline_action0_by_seed)
    if not baseline_action0_missing.empty:
        print("Missing / rejected baseline checkpoint-aligned rows:")
        display(baseline_action0_missing)
elif not baseline_action0_missing.empty:
    print(f"Baseline missing/rejected rows hidden: {len(baseline_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

plot_data = baseline_action0_by_seed.dropna(subset=["action0_fraction"]).copy()
if plot_data.empty:
    print("No checkpoint-aligned action-0 metric rows found for a0_hvg_00_baseline_s0/s1/s2.")
else:
    plot_data["seed_label"] = pd.Categorical(
        plot_data["seed_label"],
        categories=["seed 0", "seed 1", "seed 2"],
        ordered=True,
    )
    plot_data["agent"] = pd.Categorical(
        plot_data["agent"],
        categories=BASELINE_AGENTS,
        ordered=True,
    )
    baseline_action0_by_seed_audit = _display_checkpoint_audit(
        plot_data,
        title="A0 HVG baseline: action 0 / do-nothing usage at best-test checkpoint by seed",
    )

    fig_a0_hvg_baseline_action0_by_seed = px.bar(
        plot_data,
        x="agent",
        y="action0_fraction",
        color="seed_label",
        barmode="group",
        text=plot_data["action0_fraction"].map(lambda value: f"{value:.2f}"),
        hover_data={
            "run_name": True,
            "seed": True,
            "agent": True,
            "action0_fraction": ":.4f",
            "checkpoint_step_millions": ":.2f",
            "selected_metric_step_millions": ":.2f",
            "step_delta": ":.0f",
            "step_match_policy": True,
            "metric_column": True,
        },
        labels={
            "agent": "agent",
            "action0_fraction": "action 0 / do-nothing fraction",
            "seed_label": "seed",
        },
        title=(
            "A0 HVG baseline: action 0 / do-nothing usage at best-test checkpoint "
            f"({STEP_MATCH_POLICY} metric match)"
        ),
    )
    fig_a0_hvg_baseline_action0_by_seed.update_yaxes(range=[0, 1])
    fig_a0_hvg_baseline_action0_by_seed.update_traces(textposition="outside", cliponaxis=False)
    fig_a0_hvg_baseline_action0_by_seed.update_layout(
        template="plotly_white",
        height=560,
        width=1050,
        bargap=0.20,
        bargroupgap=0.06,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
        margin={"l": 70, "r": 40, "t": 110, "b": 70},
    )
    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_hvg_00_baseline_best_test_checkpoint_action0_by_agent_seed.html"
        fig_a0_hvg_baseline_action0_by_seed.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_a0_hvg_baseline_action0_by_seed.show()


## A0 HVG Baseline Seed Average

This computes the mean action-0 / do-nothing fraction across the three `a0_hvg_00_baseline` seeds, using the same best-test checkpoint-aligned rows from the previous cell.


In [ ]:
import plotly.graph_objects as go

seed_average_source = baseline_action0_by_seed.dropna(subset=["action0_fraction"]).copy()

if seed_average_source.empty:
    print("No baseline action-0 rows available to average across seeds.")
    baseline_action0_seed_average = pd.DataFrame()
else:
    baseline_action0_seed_average = (
        seed_average_source
        .groupby("agent", observed=True, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_action0_fraction=("action0_fraction", "std"),
            min_action0_fraction=("action0_fraction", "min"),
            max_action0_fraction=("action0_fraction", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_steps=("checkpoint_global_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
        .sort_values("agent")
    )
    baseline_action0_seed_average["std_action0_fraction"] = baseline_action0_seed_average["std_action0_fraction"].fillna(0.0)

    if SHOW_DIAGNOSTIC_TABLES:
        print("Best-test checkpoint-aligned action-0 average over seeds for a0_hvg_00_baseline:")
        display(baseline_action0_seed_average)

    baseline_action0_seed_average_audit = _display_checkpoint_audit(
        seed_average_source,
        title="A0 HVG baseline: mean action 0 / do-nothing usage over seeds at best-test checkpoints",
    )

    fig_a0_hvg_baseline_action0_seed_average = px.bar(
        baseline_action0_seed_average,
        x="agent",
        y="mean_action0_fraction",
        error_y="std_action0_fraction",
        text=baseline_action0_seed_average["mean_action0_fraction"].map(lambda value: f"{value:.2f}"),
        hover_data={
            "mean_action0_fraction": ":.4f",
            "std_action0_fraction": ":.4f",
            "min_action0_fraction": ":.4f",
            "max_action0_fraction": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "checkpoint_steps": True,
            "selected_metric_steps": True,
        },
        labels={
            "agent": "agent",
            "mean_action0_fraction": "mean action 0 / do-nothing fraction",
        },
        title="A0 HVG baseline: mean action 0 / do-nothing usage over seeds at best-test checkpoints",
    )
    for seed, seed_data in seed_average_source.sort_values(["seed", "agent"]).groupby("seed", sort=True):
        fig_a0_hvg_baseline_action0_seed_average.add_trace(
            go.Scatter(
                x=seed_data["agent"],
                y=seed_data["action0_fraction"],
                mode="markers",
                name=f"seed {int(seed)}",
                marker={"size": 10, "symbol": "circle", "line": {"color": "white", "width": 1}},
                customdata=seed_data[["run_name", "checkpoint_global_step", "selected_metric_step", "step_delta"]],
                hovertemplate=(
                    "run=%{customdata[0]}<br>"
                    "agent=%{x}<br>"
                    "seed action0=%{y:.4f}<br>"
                    "checkpoint_step=%{customdata[1]}<br>"
                    "selected_metric_step=%{customdata[2]}<br>"
                    "step_delta=%{customdata[3]}<extra></extra>"
                ),
            )
        )

    fig_a0_hvg_baseline_action0_seed_average.update_yaxes(range=[0, 1])
    fig_a0_hvg_baseline_action0_seed_average.update_traces(textposition="outside", selector={"type": "bar"}, cliponaxis=False)
    fig_a0_hvg_baseline_action0_seed_average.update_layout(
        template="plotly_white",
        height=560,
        width=1050,
        bargap=0.35,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
        margin={"l": 70, "r": 40, "t": 110, "b": 70},
    )
    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_hvg_00_baseline_best_test_checkpoint_action0_seed_average.html"
        fig_a0_hvg_baseline_action0_seed_average.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_a0_hvg_baseline_action0_seed_average.show()


## A0 HVG Checkpoint-Aligned Comparison Plots

These two plots use the `best_test_<run>.tar` checkpoint for each seed, then read the action-0 / do-nothing fraction from the cached evaluation metric at the checkpoint-aligned step. Bars show the mean over seeds; dots show the actual seed values. No uncertainty bars are drawn.


In [ ]:
import re

import numpy as np
import plotly.graph_objects as go
import torch

BEST_TEST_CHECKPOINT_DIR = TASK_DIR / "checkpoint" / "with_obs_stats"


def _seed_from_run_name(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else None


def _load_checkpoint_global_step(path):
    record = torch.load(path, map_location="cpu", weights_only=False)
    return int(record.get("global_step"))


A0_HVG_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_hvg_00_baseline",
        "label": "baseline",
        "plot_group": "heuristic",
        "order": 0,
    },
    {
        "family": "a0_hvg_01_eval_rho090",
        "label": "global rho heuristic",
        "plot_group": "heuristic",
        "order": 1,
    },
    {
        "family": "a0_hvg_04_eval_local_rho090",
        "label": "local rho heuristic",
        "plot_group": "heuristic",
        "order": 2,
    },
    {
        "family": "a0_hvg_02_gate_final_map",
        "label": "gate final-action MAP",
        "plot_group": "gate",
        "order": 1,
    },
    {
        "family": "a0_hvg_03_gate_hierarchical",
        "label": "gate hierarchical greedy",
        "plot_group": "gate",
        "order": 2,
    },
]
A0_HVG_CHECKPOINT_AGENTS = ["agent_0", "agent_1", "agent_2"]
A0_HVG_VARIANT_COLORS = {
    "baseline": "#1f77b4",
    "global rho heuristic": "#ff7f0e",
    "local rho heuristic": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
}
A0_AGENT_PATTERNS = {
    "agent_0": "",
    "agent_1": "/",
    "agent_2": "x",
}


def _collect_a0_hvg_best_test_checkpoint_action0_rows(variant_specs):
    history = ctx["history_wide"].copy()
    selected_runs = ctx["selected_runs"].copy()
    rows = []
    missing = []

    for spec in variant_specs:
        family = spec["family"]
        family_runs = selected_runs[
            selected_runs["run_name"].fillna("").astype(str).str.match(rf"^{re.escape(family)}_s[0-2]$")
        ].sort_values("run_name")
        if family_runs.empty:
            missing.append({"family": family, "reason": "missing_cached_runs"})
            continue

        for run in family_runs.to_dict("records"):
            run_name = str(run["run_name"])
            run_id = str(run["run_id"])
            seed = _seed_from_run_name(run_name)
            checkpoint_path = BEST_TEST_CHECKPOINT_DIR / f"best_test_{run_name}.tar"
            if not checkpoint_path.exists():
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": "missing_best_test_checkpoint",
                    "checkpoint_path": str(checkpoint_path),
                })
                continue

            try:
                checkpoint_step = _load_checkpoint_global_step(checkpoint_path)
            except Exception as exc:
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": f"checkpoint_load_failed: {type(exc).__name__}: {exc}",
                    "checkpoint_path": str(checkpoint_path),
                })
                continue

            run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
            if run_history.empty:
                missing.append({
                    "run_name": run_name,
                    "family": family,
                    "seed": seed,
                    "reason": "missing_cached_history",
                    "checkpoint_path": str(checkpoint_path),
                    "checkpoint_global_step": checkpoint_step,
                })
                continue

            for agent in A0_HVG_CHECKPOINT_AGENTS:
                values, metric_column, inverted = adm._action0_series_for_checkpoint_agent(
                    run_history,
                    str(EVAL_METRIC_SPLIT),
                    agent,
                )
                if values is None:
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "missing_agent_action0_metric",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                    })
                    continue

                selected_point = adm._select_checkpoint_metric_point(
                    run_history,
                    values,
                    checkpoint_step,
                    step_policy=STEP_MATCH_POLICY,
                )
                if selected_point is None:
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "no_metric_point_at_policy_step",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                        "metric_column": metric_column,
                    })
                    continue

                point, match_policy = selected_point
                selected_step = float(point["_step"])
                step_delta = selected_step - float(checkpoint_step)
                if MAX_STEP_DELTA is not None and abs(step_delta) > float(MAX_STEP_DELTA):
                    missing.append({
                        "run_name": run_name,
                        "family": family,
                        "seed": seed,
                        "agent": agent,
                        "reason": "step_delta_exceeds_max",
                        "checkpoint_path": str(checkpoint_path),
                        "checkpoint_global_step": checkpoint_step,
                        "selected_metric_step": selected_step,
                        "step_delta": step_delta,
                        "metric_column": metric_column,
                    })
                    continue

                rows.append({
                    "family": family,
                    "condition_label": spec["label"],
                    "plot_group": spec["plot_group"],
                    "order": spec["order"],
                    "run_name": run_name,
                    "run_id": run_id,
                    "seed": seed,
                    "agent": agent,
                    "action0_fraction": float(point["value"]),
                    "checkpoint_global_step": checkpoint_step,
                    "selected_metric_step": selected_step,
                    "step_delta": step_delta,
                    "step_match_policy": match_policy,
                    "metric_column": metric_column,
                    "metric_inverted": bool(inverted),
                    "checkpoint_path": str(checkpoint_path),
                })

    return pd.DataFrame(rows), pd.DataFrame(missing)


def _checkpoint_action0_summary(seed_rows):
    if seed_rows.empty:
        return pd.DataFrame()
    return (
        seed_rows
        .groupby(["plot_group", "condition_label", "order", "agent"], observed=True, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_steps=("checkpoint_global_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
        .sort_values(["plot_group", "order", "agent"])
    )


def _plot_checkpoint_action0_grouped_bars(seed_rows, *, plot_group, labels, title, save_name):
    plot_seed_rows = seed_rows[
        seed_rows["condition_label"].isin(labels)
        & (seed_rows["plot_group"].eq(plot_group) | seed_rows["condition_label"].eq("baseline"))
    ].copy()
    plot_seed_rows["plot_group"] = plot_group
    if plot_seed_rows.empty:
        print(f"No checkpoint-aligned action-0 rows available for {plot_group}.")
        return None, pd.DataFrame()

    plot_seed_rows["condition_label"] = pd.Categorical(plot_seed_rows["condition_label"], categories=labels, ordered=True)
    plot_seed_rows["agent"] = pd.Categorical(plot_seed_rows["agent"].astype(str), categories=A0_HVG_CHECKPOINT_AGENTS, ordered=True)
    summary = _checkpoint_action0_summary(plot_seed_rows)
    summary["condition_label"] = pd.Categorical(summary["condition_label"], categories=labels, ordered=True)
    summary["agent"] = pd.Categorical(summary["agent"].astype(str), categories=A0_HVG_CHECKPOINT_AGENTS, ordered=True)
    summary = summary.sort_values(["condition_label", "agent"])

    audit = _display_checkpoint_audit(
        plot_seed_rows,
        title=title,
        labels=labels,
        plot_group=plot_group,
    )

    if SHOW_DIAGNOSTIC_TABLES:
        print(title)
        display(summary)

    x_index = {label: idx for idx, label in enumerate(labels)}
    agent_width = 0.22
    agent_offsets = {
        agent: (idx - (len(A0_HVG_CHECKPOINT_AGENTS) - 1) / 2.0) * agent_width
        for idx, agent in enumerate(A0_HVG_CHECKPOINT_AGENTS)
    }
    bar_width = agent_width * 0.86

    fig = go.Figure()
    for label in labels:
        label_summary = summary[summary["condition_label"].astype(str).eq(label)].set_index("agent")
        x_values = []
        y_values = []
        text_values = []
        customdata = []
        pattern_shapes = []
        hover_agents = []
        for agent in A0_HVG_CHECKPOINT_AGENTS:
            x_values.append(x_index[label] + agent_offsets[agent])
            pattern_shapes.append(A0_AGENT_PATTERNS[agent])
            hover_agents.append(agent)
            if agent in label_summary.index:
                row = label_summary.loc[agent]
                y = float(row["mean_action0_fraction"])
                y_values.append(y)
                text_values.append(f"{y:.2f}")
                customdata.append([
                    agent,
                    int(row["n_seeds"]),
                    str(row["seeds"]),
                    str(row["checkpoint_steps"]),
                    str(row["selected_metric_steps"]),
                    str(row["runs"]),
                ])
            else:
                y_values.append(None)
                text_values.append("")
                customdata.append([agent, 0, "[]", "[]", "[]", "[]"])

        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                width=bar_width,
                name=label,
                legendgroup=label,
                marker={
                    "color": A0_HVG_VARIANT_COLORS[label],
                    "line": {"color": "rgba(0,0,0,0.35)", "width": 0.8},
                    "pattern": {"shape": pattern_shapes, "fgcolor": "rgba(255,255,255,0.75)", "size": 8},
                },
                text=text_values,
                textposition="outside",
                cliponaxis=False,
                customdata=np.array(customdata, dtype=object),
                hovertemplate=(
                    f"run type={label}<br>"
                    "agent=%{customdata[0]}<br>"
                    "mean action-0 fraction=%{y:.4f}<br>"
                    "n_seeds=%{customdata[1]}<br>"
                    "seeds=%{customdata[2]}<br>"
                    "checkpoint_steps=%{customdata[3]}<br>"
                    "selected_metric_steps=%{customdata[4]}<br>"
                    "runs=%{customdata[5]}<extra></extra>"
                ),
                hovertext=hover_agents,
            )
        )

    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square"}
    for (label, agent), group in plot_seed_rows.groupby(["condition_label", "agent"], observed=True, sort=False):
        label = str(label)
        agent = str(agent)
        group = group.sort_values("seed").copy()
        if group.empty:
            continue
        jitter = np.linspace(-bar_width * 0.22, bar_width * 0.22, len(group)) if len(group) > 1 else np.array([0.0])
        x_center = x_index[label] + agent_offsets[agent]
        fig.add_trace(
            go.Scatter(
                x=x_center + jitter,
                y=group["action0_fraction"],
                mode="markers",
                name=f"{label} {agent} seeds",
                legendgroup=label,
                showlegend=False,
                marker={
                    "color": A0_HVG_VARIANT_COLORS[label],
                    "size": 11,
                    "symbol": [seed_marker_symbols.get(int(seed), "circle") for seed in group["seed"]],
                    "line": {"color": "white", "width": 1.2},
                    "opacity": 0.95,
                },
                customdata=group[[
                    "run_name",
                    "seed",
                    "checkpoint_global_step",
                    "selected_metric_step",
                    "step_delta",
                    "metric_column",
                ]].to_numpy(dtype=object),
                hovertemplate=(
                    "run=%{customdata[0]}<br>"
                    "seed=%{customdata[1]}<br>"
                    f"agent={agent}<br>"
                    f"run type={label}<br>"
                    "action-0 fraction=%{y:.4f}<br>"
                    "checkpoint_step=%{customdata[2]}<br>"
                    "selected_metric_step=%{customdata[3]}<br>"
                    "step_delta=%{customdata[4]}<br>"
                    "metric=%{customdata[5]}<extra></extra>"
                ),
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(labels))),
        ticktext=labels,
        title_text="run type",
    )
    fig.update_yaxes(range=[0, 1], title_text="action-0 / do-nothing fraction")
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=620,
        width=1250,
        bargap=0.20,
        legend={
            "title": {"text": "run variant"},
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 15},
        },
        margin={"l": 75, "r": 180, "t": 95, "b": 120},
    )
    fig.add_annotation(
        text="Colors match episodic-survival run variants. Bar patterns: agent_0=solid, agent_1=slash, agent_2=cross. Dots: s0=circle, s1=diamond, s2=square.",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.20,
        showarrow=False,
        align="left",
        font={"size": 12, "color": "#555"},
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / f"{save_name}.html"
        fig.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig.show()
    return fig, summary


hvg_checkpoint_action0_by_seed, hvg_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
    A0_HVG_CHECKPOINT_VARIANTS
)
print(f"A0 HVG best-test checkpoint-aligned action-0 rows: {len(hvg_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(hvg_checkpoint_action0_by_seed.sort_values(["order", "condition_label", "seed", "agent"]))
    if not hvg_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 HVG checkpoint-aligned rows:")
        display(hvg_checkpoint_action0_missing)
elif not hvg_checkpoint_action0_missing.empty:
    print(f"A0 HVG missing/skipped rows hidden: {len(hvg_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_hvg_checkpoint_action0_heuristics, a0_hvg_checkpoint_action0_heuristics_summary = _plot_checkpoint_action0_grouped_bars(
    hvg_checkpoint_action0_by_seed,
    plot_group="heuristic",
    labels=["baseline", "global rho heuristic", "local rho heuristic"],
    title="A0 HVG: checkpoint-aligned action-0 fraction, baseline vs heuristic overrides",
    save_name="a0_hvg_checkpoint_action0_baseline_global_local_heuristic",
)

fig_a0_hvg_checkpoint_action0_gates, a0_hvg_checkpoint_action0_gates_summary = _plot_checkpoint_action0_grouped_bars(
    hvg_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=["baseline", "gate final-action MAP", "gate hierarchical greedy"],
    title="A0 HVG: checkpoint-aligned action-0 fraction, baseline vs gate variants",
    save_name="a0_hvg_checkpoint_action0_baseline_gate_map_hierarchical",
)


## A0 Sparse16 Checkpoint-Aligned Action-0 Plots

Same checkpoint-aligned action-0 / do-nothing plot style as above, but for the Sparse16 intervention-penalty runs. Bars show mean over seeds at each run's `best_test_<run>.tar` checkpoint; dots show the individual seed values. Flat and gated policies are separated into two plots.


In [ ]:
A0_SPARSE16_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_sparse16_flat_p000",
        "label": "flat p0.000",
        "plot_group": "flat",
        "order": 0,
    },
    {
        "family": "a0_sparse16_flat_p001",
        "label": "flat p0.001",
        "plot_group": "flat",
        "order": 1,
    },
    {
        "family": "a0_sparse16_flat_p003",
        "label": "flat p0.003",
        "plot_group": "flat",
        "order": 2,
    },
    {
        "family": "a0_sparse16_flat_p010",
        "label": "flat p0.010",
        "plot_group": "flat",
        "order": 3,
    },
    {
        "family": "a0_sparse16_gated_p000",
        "label": "gated p0.000",
        "plot_group": "gated",
        "order": 0,
    },
    {
        "family": "a0_sparse16_gated_p001",
        "label": "gated p0.001",
        "plot_group": "gated",
        "order": 1,
    },
    {
        "family": "a0_sparse16_gated_p003",
        "label": "gated p0.003",
        "plot_group": "gated",
        "order": 2,
    },
    {
        "family": "a0_sparse16_gated_p010",
        "label": "gated p0.010",
        "plot_group": "gated",
        "order": 3,
    },
]

A0_SPARSE16_VARIANT_COLORS = {
    "flat p0.000": "#1f77b4",
    "flat p0.001": "#ff7f0e",
    "flat p0.003": "#2ca02c",
    "flat p0.010": "#d62728",
    "gated p0.000": "#1f77b4",
    "gated p0.001": "#ff7f0e",
    "gated p0.003": "#2ca02c",
    "gated p0.010": "#d62728",
}

# The shared plotting helper above reads this color map by label.
A0_HVG_VARIANT_COLORS.update(A0_SPARSE16_VARIANT_COLORS)

# Sparse16 exists in both CPU and GPU run-data folders. Keep one source slice so
# identical run names are not counted twice. If RUN_DATA_SOURCE is already set to
# "cpu" or "gpu" at the top of the notebook, this follows that setting.
SPARSE16_CHECKPOINT_RUN_DATA_SOURCE = adm._normalize_run_data_source(RUN_DATA_SOURCE) or "gpu"
_sparse16_selected_runs_original = ctx["selected_runs"]
_sparse16_selected_runs = adm._filter_rows_by_run_data_source(
    _sparse16_selected_runs_original,
    SPARSE16_CHECKPOINT_RUN_DATA_SOURCE,
    keep_unsplit=False,
)
if _sparse16_selected_runs.empty:
    print(
        f"No Sparse16 rows found for source={SPARSE16_CHECKPOINT_RUN_DATA_SOURCE!r}; "
        "falling back to all loaded Sparse16 rows."
    )
    _sparse16_selected_runs = _sparse16_selected_runs_original
else:
    _sparse16_source_mask = _sparse16_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_sparse16_")
    print(
        f"Sparse16 checkpoint plots using run-data source={SPARSE16_CHECKPOINT_RUN_DATA_SOURCE!r} "
        f"({_sparse16_source_mask.sum()} cached Sparse16 runs, "
        f"{_sparse16_selected_runs.loc[_sparse16_source_mask, 'run_name'].nunique()} unique Sparse16 run names)."
    )

ctx["selected_runs"] = _sparse16_selected_runs
try:
    sparse16_checkpoint_action0_by_seed, sparse16_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
        A0_SPARSE16_CHECKPOINT_VARIANTS
    )
finally:
    ctx["selected_runs"] = _sparse16_selected_runs_original
print(f"A0 Sparse16 best-test checkpoint-aligned action-0 rows: {len(sparse16_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(sparse16_checkpoint_action0_by_seed.sort_values(["plot_group", "order", "seed", "agent"]))
    if not sparse16_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 Sparse16 checkpoint-aligned rows:")
        display(sparse16_checkpoint_action0_missing)
elif not sparse16_checkpoint_action0_missing.empty:
    print(f"A0 Sparse16 missing/skipped rows hidden: {len(sparse16_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_sparse16_checkpoint_action0_flat, a0_sparse16_checkpoint_action0_flat_summary = _plot_checkpoint_action0_grouped_bars(
    sparse16_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=["flat p0.000", "flat p0.001", "flat p0.003", "flat p0.010"],
    title="A0 Sparse16 flat: checkpoint-aligned action-0 fraction by intervention penalty",
    save_name="a0_sparse16_checkpoint_action0_flat_penalties",
)

fig_a0_sparse16_checkpoint_action0_gated, a0_sparse16_checkpoint_action0_gated_summary = _plot_checkpoint_action0_grouped_bars(
    sparse16_checkpoint_action0_by_seed,
    plot_group="gated",
    labels=["gated p0.000", "gated p0.001", "gated p0.003", "gated p0.010"],
    title="A0 Sparse16 gated: checkpoint-aligned action-0 fraction by intervention penalty",
    save_name="a0_sparse16_checkpoint_action0_gated_penalties",
)

# display(fig_a0_sparse16_checkpoint_action0_flat)
# fig_a0_sparse16_checkpoint_action0_gated


## A0 AIB Checkpoint-Aligned Action-0 Plots

Same checkpoint-aligned action-0 / do-nothing plot style as above, but for the Adaptive Intervention Budget runs. Bars show mean over seeds at each run's `best_test_<run>.tar` checkpoint; dots show the individual seed values. The first plot compares flat budget variants against the plain baseline; the second compares the gated AIB variant against the matched flat local target `0.20` and baseline.


In [ ]:
A0_AIB_CHECKPOINT_VARIANTS = [
    {
        "family": "a0_hvg_00_baseline",
        "label": "baseline",
        "plot_group": "flat",
        "order": 0,
    },
    {
        "family": "a0_aib_00_flat_local_t020",
        "label": "flat local t0.20",
        "plot_group": "flat",
        "order": 1,
    },
    {
        "family": "a0_aib_01_flat_local_t010",
        "label": "flat local t0.10",
        "plot_group": "flat",
        "order": 2,
    },
    {
        "family": "a0_aib_02_flat_local_t035",
        "label": "flat local t0.35",
        "plot_group": "flat",
        "order": 3,
    },
    {
        "family": "a0_aib_04_flat_nonidle_t020",
        "label": "flat non-idle t0.20",
        "plot_group": "flat",
        "order": 4,
    },
    {
        "family": "a0_aib_00_flat_local_t020",
        "label": "flat local t0.20",
        "plot_group": "gate",
        "order": 1,
    },
    {
        "family": "a0_aib_03_gate_hgreedy_sep_local_t020",
        "label": "gate h-greedy t0.20",
        "plot_group": "gate",
        "order": 2,
    },
]

A0_AIB_VARIANT_COLORS = {
    "baseline": "#1f77b4",
    "flat local t0.20": "#ff7f0e",
    "flat local t0.10": "#2ca02c",
    "flat local t0.35": "#d62728",
    "flat non-idle t0.20": "#9467bd",
    "gate h-greedy t0.20": "#8c564b",
}

# The shared plotting helper above reads this color map by label.
A0_HVG_VARIANT_COLORS.update(A0_AIB_VARIANT_COLORS)

# AIB exists in both CPU and GPU run-data folders. Keep one source slice so
# identical run names are not counted twice. Unsplit folders, such as the plain
# A0 HVG baseline, are kept as shared baselines.
AIB_CHECKPOINT_RUN_DATA_SOURCE = adm._normalize_run_data_source(RUN_DATA_SOURCE) or "gpu"
_aib_selected_runs_original = ctx["selected_runs"]
_aib_selected_runs = adm._filter_rows_by_run_data_source(
    _aib_selected_runs_original,
    AIB_CHECKPOINT_RUN_DATA_SOURCE,
    keep_unsplit=True,
)
if _aib_selected_runs.empty:
    print(
        f"No AIB rows found for source={AIB_CHECKPOINT_RUN_DATA_SOURCE!r}; "
        "falling back to all loaded rows."
    )
    _aib_selected_runs = _aib_selected_runs_original
else:
    _aib_source_mask = _aib_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_aib_")
    _baseline_source_mask = _aib_selected_runs["run_name"].fillna("").astype(str).str.startswith("a0_hvg_00_baseline_")
    print(
        f"AIB checkpoint plots using run-data source={AIB_CHECKPOINT_RUN_DATA_SOURCE!r} "
        f"({_aib_source_mask.sum()} cached AIB runs, "
        f"{_aib_selected_runs.loc[_aib_source_mask, 'run_name'].nunique()} unique AIB run names; "
        f"{_baseline_source_mask.sum()} cached plain-baseline runs)."
    )

ctx["selected_runs"] = _aib_selected_runs
try:
    aib_checkpoint_action0_by_seed, aib_checkpoint_action0_missing = _collect_a0_hvg_best_test_checkpoint_action0_rows(
        A0_AIB_CHECKPOINT_VARIANTS
    )
finally:
    ctx["selected_runs"] = _aib_selected_runs_original
print(f"A0 AIB best-test checkpoint-aligned action-0 rows: {len(aib_checkpoint_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(aib_checkpoint_action0_by_seed.sort_values(["plot_group", "order", "seed", "agent"]))
    if not aib_checkpoint_action0_missing.empty:
        print("Missing / skipped A0 AIB checkpoint-aligned rows:")
        display(aib_checkpoint_action0_missing)
elif not aib_checkpoint_action0_missing.empty:
    print(f"A0 AIB missing/skipped rows hidden: {len(aib_checkpoint_action0_missing)}. Set SHOW_DIAGNOSTIC_TABLES=True to inspect them.")

fig_a0_aib_checkpoint_action0_flat, a0_aib_checkpoint_action0_flat_summary = _plot_checkpoint_action0_grouped_bars(
    aib_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=["baseline", "flat local t0.10", "flat local t0.20", "flat local t0.35", "flat non-idle t0.20"],
    title="A0 AIB flat budget variants: checkpoint-aligned action-0 fraction",
    save_name="a0_aib_checkpoint_action0_flat_budget_variants",
)

fig_a0_aib_checkpoint_action0_gate, a0_aib_checkpoint_action0_gate_summary = _plot_checkpoint_action0_grouped_bars(
    aib_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=["baseline", "flat local t0.20", "gate h-greedy t0.20"],
    title="A0 AIB gated budget variant: checkpoint-aligned action-0 fraction",
    save_name="a0_aib_checkpoint_action0_gate_budget_variant",
)

# display(fig_a0_aib_checkpoint_action0_flat)
# fig_a0_aib_checkpoint_action0_gate


## Teacher-Student BC Action-0 Usage by Agent

Teacher-student checkpoints are offline BC checkpoints, not normal W&B training runs, so they do not have the same rollout action-distribution history as MAPPO runs. This plot uses the final BC eval metric saved in `teacher_student_epoch_metrics.csv`: `action0_fraction = 1 - pred_nonidle_frac`. Bars show the mean over seeds; dots show the actual seed values.


In [ ]:
TEACHER_STUDENT_EPOCH_METRICS_PATH = TASK_DIR / "outputs" / "comparison_dashboard_figures" / "teacher_student_epoch_metrics.csv"

TEACHER_STUDENT_FAMILY_LABELS = {
    "local_bc": "BC bal0.50 w5 aux0.50",
    "local_bc_bal020_w3_aux025": "BC bal0.20 w3 aux0.25",
}
TEACHER_STUDENT_FAMILY_ORDER = {
    "local_bc": 0,
    "local_bc_bal020_w3_aux025": 1,
}
TEACHER_STUDENT_COLORS = {
    "BC bal0.50 w5 aux0.50": "#1f77b4",
    "BC bal0.20 w3 aux0.25": "#ff7f0e",
}
A0_HVG_VARIANT_COLORS.update(TEACHER_STUDENT_COLORS)

if not TEACHER_STUDENT_EPOCH_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Teacher-student epoch metrics not found: {TEACHER_STUDENT_EPOCH_METRICS_PATH}"
    )

teacher_student_epoch_metrics = pd.read_csv(TEACHER_STUDENT_EPOCH_METRICS_PATH)
required_columns = {
    "checkpoint",
    "checkpoint_family",
    "checkpoint_seed",
    "phase",
    "epoch",
    "agent",
    "pred_nonidle_frac",
}
missing_columns = required_columns.difference(teacher_student_epoch_metrics.columns)
if missing_columns:
    raise KeyError(f"Missing teacher-student epoch metric columns: {sorted(missing_columns)}")

teacher_student_eval = teacher_student_epoch_metrics[
    teacher_student_epoch_metrics["phase"].astype(str).eq("eval")
].copy()
teacher_student_eval["epoch"] = pd.to_numeric(teacher_student_eval["epoch"], errors="coerce")
teacher_student_eval["pred_nonidle_frac"] = pd.to_numeric(
    teacher_student_eval["pred_nonidle_frac"], errors="coerce"
)
teacher_student_eval = teacher_student_eval.dropna(
    subset=["epoch", "pred_nonidle_frac", "checkpoint_family", "checkpoint_seed", "agent"]
)

if teacher_student_eval.empty:
    raise RuntimeError("No teacher-student eval rows with pred_nonidle_frac were found.")

_final_epoch_by_checkpoint = teacher_student_eval.groupby("checkpoint", dropna=False)["epoch"].transform("max")
teacher_student_final = teacher_student_eval[teacher_student_eval["epoch"].eq(_final_epoch_by_checkpoint)].copy()
teacher_student_final["condition_label"] = teacher_student_final["checkpoint_family"].map(TEACHER_STUDENT_FAMILY_LABELS).fillna(
    teacher_student_final["checkpoint_family"].astype(str)
)
teacher_student_final["order"] = teacher_student_final["checkpoint_family"].map(TEACHER_STUDENT_FAMILY_ORDER).fillna(99).astype(int)
teacher_student_final["plot_group"] = "teacher_student"
teacher_student_final["seed"] = pd.to_numeric(teacher_student_final["checkpoint_seed"], errors="coerce").astype("Int64")
teacher_student_final["action0_fraction"] = 1.0 - teacher_student_final["pred_nonidle_frac"]
teacher_student_final["run_name"] = teacher_student_final["checkpoint"].astype(str).str.replace(r"\.tar$", "", regex=True)
teacher_student_final["run_id"] = teacher_student_final["run_name"]
_teacher_student_optimizer_steps = teacher_student_final.get(
    "optimizer_steps",
    pd.Series([pd.NA] * len(teacher_student_final), index=teacher_student_final.index),
)
teacher_student_final["checkpoint_global_step"] = _teacher_student_optimizer_steps.values
teacher_student_final["selected_metric_step"] = _teacher_student_optimizer_steps.values
teacher_student_final["step_delta"] = 0
teacher_student_final["metric_column"] = "teacher_student_epoch_metrics.pred_nonidle_frac"
teacher_student_final["metric_inverted"] = True


def _teacher_student_checkpoint_path(checkpoint_name):
    candidate = TASK_DIR / "checkpoint" / "teacher_student" / str(checkpoint_name)
    return str(candidate if candidate.exists() else checkpoint_name)


teacher_student_final["checkpoint_path"] = teacher_student_final["checkpoint"].map(_teacher_student_checkpoint_path)

teacher_student_action0_by_seed = teacher_student_final[[
    "run_name",
    "run_id",
    "checkpoint",
    "checkpoint_family",
    "condition_label",
    "plot_group",
    "order",
    "seed",
    "agent",
    "action0_fraction",
    "pred_nonidle_frac",
    "teacher_nonidle_frac",
    "accuracy",
    "false_noop_rate",
    "false_intervention_rate",
    "checkpoint_global_step",
    "selected_metric_step",
    "step_delta",
    "metric_column",
    "metric_inverted",
    "checkpoint_path",
]].copy()

print(f"Teacher-student final-epoch BC eval action-0 rows: {len(teacher_student_action0_by_seed)}")
if SHOW_DIAGNOSTIC_TABLES:
    display(teacher_student_action0_by_seed.sort_values(["order", "seed", "agent"]))

fig_teacher_student_action0_by_agent, teacher_student_action0_summary = _plot_checkpoint_action0_grouped_bars(
    teacher_student_action0_by_seed,
    plot_group="teacher_student",
    labels=["BC bal0.50 w5 aux0.50", "BC bal0.20 w3 aux0.25"],
    title="Teacher-student BC: final-epoch eval action-0 fraction by agent",
    save_name="teacher_student_bc_final_eval_action0_by_agent",
)

fig_teacher_student_action0_by_agent


## Checkpoint-Aligned Action-0 / Survival Tradeoff

These plots reproduce the Action-0 / Survival Tradeoff idea from the CPU action-distribution notebook, but they use the checkpoint-aligned `best_test_<run>.tar` rows built above. Each seed point is the mean action-0 fraction across agents at that checkpoint versus the episodic survival metric selected at the same checkpoint step. Diamonds show the condition mean over seeds.


In [ ]:
SURVIVAL_METRIC_CANDIDATES = [
    "test/charts/episodic_survival",
    "test/episodic_survival",
    "validation/episodic_survival",
    "charts/episodic_survival",
    "train_eval/charts/episodic_survival",
    "train_eval/episodic_survival",
]


def _checkpoint_tradeoff_survival_scale(values):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def _plot_range_with_padding(values, *, lower=None, upper=None, pad_fraction=0.12, min_span=0.05):
    numeric = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if numeric.empty:
        return [lower, upper] if lower is not None and upper is not None else None

    vmin = float(numeric.min())
    vmax = float(numeric.max())
    span = max(vmax - vmin, float(min_span))
    pad = span * float(pad_fraction)
    lo = vmin - pad
    hi = vmax + pad

    if lower is not None:
        lo = max(float(lower), lo)
    if upper is not None:
        hi = min(float(upper), hi)
    if hi <= lo:
        hi = lo + float(min_span)
        if upper is not None and hi > float(upper):
            hi = float(upper)
            lo = max(float(lower) if lower is not None else hi - float(min_span), hi - float(min_span))
    return [lo, hi]


def _checkpoint_tradeoff_filter_rows(seed_rows, *, labels=None, plot_group=None):
    if seed_rows is None or seed_rows.empty:
        return pd.DataFrame()
    data = seed_rows.copy()
    if labels is not None and "condition_label" in data.columns:
        data = data[data["condition_label"].astype(str).isin([str(label) for label in labels])].copy()
    if plot_group is not None and "plot_group" in data.columns:
        baseline_mask = data.get("condition_label", pd.Series(index=data.index, dtype=object)).astype(str).eq("baseline")
        data = data[data["plot_group"].astype(str).eq(str(plot_group)) | baseline_mask].copy()
        data["plot_group"] = str(plot_group)
    return data


def _checkpoint_survival_point_for_run(run_history, checkpoint_step):
    if run_history is None or run_history.empty:
        return None
    for metric in SURVIVAL_METRIC_CANDIDATES:
        if metric not in run_history.columns:
            continue
        values = pd.to_numeric(run_history[metric], errors="coerce")
        if not values.notna().any():
            continue
        selected = adm._select_checkpoint_metric_point(
            run_history,
            values,
            checkpoint_step,
            step_policy=STEP_MATCH_POLICY,
        )
        if selected is None:
            continue
        point, match_policy = selected
        scale = _checkpoint_tradeoff_survival_scale(values)
        selected_step = float(point["_step"])
        return {
            "survival_pct": float(point["value"]) * scale,
            "survival_metric": metric,
            "survival_selected_metric_step": selected_step,
            "survival_selected_metric_step_millions": selected_step / 1_000_000,
            "survival_step_delta": selected_step - float(checkpoint_step),
            "survival_step_match_policy": match_policy,
        }
    return None


def _checkpoint_action0_survival_tradeoff_rows(seed_rows, *, labels=None, plot_group=None):
    plot_rows = _checkpoint_tradeoff_filter_rows(seed_rows, labels=labels, plot_group=plot_group)
    if plot_rows.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    plot_rows["action0_fraction"] = pd.to_numeric(plot_rows["action0_fraction"], errors="coerce")
    group_cols = [
        "plot_group",
        "condition_label",
        "order",
        "run_name",
        "run_id",
        "seed",
        "checkpoint_path",
        "checkpoint_global_step",
        "selected_metric_step",
        "step_delta",
        "step_match_policy",
    ]
    group_cols = [col for col in group_cols if col in plot_rows.columns]
    per_run = (
        plot_rows
        .dropna(subset=["action0_fraction"])
        .groupby(group_cols, dropna=False, as_index=False)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_agent_action0_fraction=("action0_fraction", "std"),
            n_agents=("agent", "nunique"),
            agents=("agent", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        )
    )
    if per_run.empty:
        return per_run, pd.DataFrame(), pd.DataFrame()
    per_run["std_agent_action0_fraction"] = per_run["std_agent_action0_fraction"].fillna(0.0)
    per_run["checkpoint_file"] = per_run["checkpoint_path"].fillna("").astype(str).map(lambda value: Path(value).name if value else "")

    history = ctx["history_wide"].copy()
    survival_rows = []
    missing_rows = []
    for row in per_run.to_dict("records"):
        run_id = str(row.get("run_id", ""))
        checkpoint_step = row.get("checkpoint_global_step")
        run_history = history[history["run_id"].astype(str).eq(run_id)].sort_values("_step").copy()
        survival = _checkpoint_survival_point_for_run(run_history, checkpoint_step)
        if survival is None:
            missing_rows.append({
                "condition_label": row.get("condition_label"),
                "run_name": row.get("run_name"),
                "run_id": row.get("run_id"),
                "seed": row.get("seed"),
                "checkpoint_path": row.get("checkpoint_path"),
                "checkpoint_global_step": row.get("checkpoint_global_step"),
                "reason": "missing_checkpoint_survival_metric",
            })
            continue
        survival_rows.append({**row, **survival})

    tradeoff = pd.DataFrame(survival_rows)
    missing = pd.DataFrame(missing_rows)
    if tradeoff.empty:
        return tradeoff, pd.DataFrame(), missing

    agg = (
        tradeoff
        .groupby(["plot_group", "condition_label", "order"], dropna=False, as_index=False)
        .agg(
            mean_action0_fraction=("mean_action0_fraction", "mean"),
            std_action0_fraction=("mean_action0_fraction", "std"),
            mean_survival_pct=("survival_pct", "mean"),
            std_survival_pct=("survival_pct", "std"),
            n_seeds=("run_id", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            checkpoint_files=("checkpoint_file", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            survival_metrics=("survival_metric", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
        )
        .sort_values(["plot_group", "order", "condition_label"])
    )
    agg["std_action0_fraction"] = agg["std_action0_fraction"].fillna(0.0)
    agg["std_survival_pct"] = agg["std_survival_pct"].fillna(0.0)
    return tradeoff, agg, missing


def _display_checkpoint_tradeoff_audit(tradeoff, agg, *, title):
    if tradeoff.empty:
        return pd.DataFrame()
    mean_cols = agg[[
        "plot_group",
        "condition_label",
        "mean_action0_fraction",
        "mean_survival_pct",
        "n_seeds",
        "seeds",
    ]].rename(columns={
        "mean_action0_fraction": "condition_mean_plotted_action0_fraction",
        "mean_survival_pct": "condition_mean_plotted_survival_pct",
    })
    audit = tradeoff.merge(mean_cols, on=["plot_group", "condition_label"], how="left")
    audit = audit.rename(columns={
        "mean_action0_fraction": "seed_plotted_action0_fraction",
        "survival_pct": "seed_plotted_survival_pct",
    })
    keep = [
        "condition_label",
        "run_name",
        "seed",
        "checkpoint_file",
        "checkpoint_global_step",
        "checkpoint_path",
        "seed_plotted_action0_fraction",
        "seed_plotted_survival_pct",
        "condition_mean_plotted_action0_fraction",
        "condition_mean_plotted_survival_pct",
        "n_seeds",
        "seeds",
        "survival_metric",
        "selected_metric_step",
        "survival_selected_metric_step",
        "step_delta",
        "survival_step_delta",
    ]
    keep = [col for col in keep if col in audit.columns]
    audit = audit[keep].sort_values(["condition_label", "seed", "run_name"], kind="stable").reset_index(drop=True)
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print(f"Checkpoint tradeoff values used for: {title}")
        display(audit)
    return audit


def _plot_checkpoint_action0_survival_tradeoff(seed_rows, *, plot_group, labels, title, save_name):
    tradeoff, agg, missing = _checkpoint_action0_survival_tradeoff_rows(
        seed_rows,
        labels=labels,
        plot_group=plot_group,
    )
    if not missing.empty:
        print(f"{title}: missing survival for {len(missing)} checkpoint/run rows.")
        if SHOW_DIAGNOSTIC_TABLES:
            display(missing)
    if tradeoff.empty or agg.empty:
        print(f"No checkpoint-aligned survival/action-0 tradeoff data available for: {title}")
        return None, tradeoff, agg, missing

    audit = _display_checkpoint_tradeoff_audit(tradeoff, agg, title=title)
    label_order = [label for label in labels if label in set(agg["condition_label"].astype(str))]
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {
        label: A0_HVG_VARIANT_COLORS.get(label, palette[idx % len(palette)])
        for idx, label in enumerate(label_order)
    }

    fig = go.Figure()
    for label in label_order:
        color = colors[label]
        seed_points = tradeoff[tradeoff["condition_label"].astype(str).eq(label)].sort_values("seed")
        if not seed_points.empty:
            fig.add_trace(
                go.Scatter(
                    x=seed_points["mean_action0_fraction"],
                    y=seed_points["survival_pct"],
                    mode="markers",
                    name=f"{label} seeds",
                    legendgroup=label,
                    showlegend=False,
                    marker={
                        "color": color,
                        "size": 5.5,
                        "opacity": 0.48,
                        "line": {"color": "white", "width": 0.9},
                    },
                    customdata=np.stack([
                        seed_points["run_name"].astype(str),
                        seed_points["seed"],
                        seed_points["checkpoint_file"].astype(str),
                        seed_points["checkpoint_global_step"],
                        seed_points["survival_metric"].astype(str),
                        seed_points["agents"].astype(str),
                    ], axis=-1),
                    hovertemplate=(
                        "run=%{customdata[0]}<br>"
                        "seed=%{customdata[1]}<br>"
                        "checkpoint=%{customdata[2]}<br>"
                        "checkpoint_step=%{customdata[3]}<br>"
                        "mean action-0=%{x:.4f}<br>"
                        "survival=%{y:.2f}%<br>"
                        "survival metric=%{customdata[4]}<br>"
                        "agents=%{customdata[5]}<extra></extra>"
                    ),
                )
            )

        mean_row = agg[agg["condition_label"].astype(str).eq(label)].iloc[0]
        fig.add_trace(
            go.Scatter(
                x=[mean_row["mean_action0_fraction"]],
                y=[mean_row["mean_survival_pct"]],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=True,
                marker={
                    "color": color,
                    "size": 12,
                    "symbol": "diamond",
                    "line": {"color": "black", "width": 1.1},
                },
                error_x={"type": "data", "array": [float(mean_row["std_action0_fraction"])]},
                error_y={"type": "data", "array": [float(mean_row["std_survival_pct"])]},
                customdata=np.array([[
                    mean_row["n_seeds"],
                    str(mean_row["seeds"]),
                    str(mean_row["runs"]),
                    str(mean_row["survival_metrics"]),
                ]], dtype=object),
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    "mean action-0=%{x:.4f}<br>"
                    "mean survival=%{y:.2f}%<br>"
                    "n_seeds=%{customdata[0]}<br>"
                    "seeds=%{customdata[1]}<br>"
                    "runs=%{customdata[2]}<br>"
                    "survival metrics=%{customdata[3]}<extra></extra>"
                ),
            )
        )

    fig.add_annotation(
        text="better: higher survival and more action 0",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-95,
        ay=45,
        font={"size": 12, "color": "#333"},
        arrowcolor="#333",
    )
    fig.update_layout(
        title=f"{title}<br><sup>seed dots; diamond/error bars = mean/std over seeds; values aligned to best-test checkpoint step</sup>",
        template="plotly_white",
        height=680,
        width=1200,
        hovermode="closest",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 340, "t": 105, "b": 70},
    )
    tradeoff_x_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(tradeoff["mean_action0_fraction"], errors="coerce"),
            pd.to_numeric(agg["mean_action0_fraction"], errors="coerce"),
        ], ignore_index=True),
        lower=0,
        upper=1,
        min_span=0.08,
    )
    tradeoff_y_range = _plot_range_with_padding(
        pd.concat([
            pd.to_numeric(tradeoff["survival_pct"], errors="coerce"),
            pd.to_numeric(agg["mean_survival_pct"], errors="coerce"),
        ], ignore_index=True),
        lower=0,
        upper=105,
        min_span=8,
    )
    fig.update_xaxes(title_text="mean action-0 fraction across agents", range=tradeoff_x_range)
    fig.update_yaxes(title_text="episodic survival (%)", range=tradeoff_y_range)

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / f"{save_name}.html"
        fig.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig.show()
    return fig, tradeoff, agg, missing


fig_hvg_tradeoff_heuristic, hvg_tradeoff_heuristic_rows, hvg_tradeoff_heuristic_summary, hvg_tradeoff_heuristic_missing = _plot_checkpoint_action0_survival_tradeoff(
    hvg_checkpoint_action0_by_seed,
    plot_group="heuristic",
    labels=["baseline", "global rho heuristic", "local rho heuristic"],
    title="A0 HVG: checkpoint-aligned action-0 / survival tradeoff, baseline vs heuristic overrides",
    save_name="a0_hvg_checkpoint_action0_survival_tradeoff_heuristic",
)

fig_hvg_tradeoff_gate, hvg_tradeoff_gate_rows, hvg_tradeoff_gate_summary, hvg_tradeoff_gate_missing = _plot_checkpoint_action0_survival_tradeoff(
    hvg_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=["baseline", "gate final-action MAP", "gate hierarchical greedy"],
    title="A0 HVG: checkpoint-aligned action-0 / survival tradeoff, baseline vs gate variants",
    save_name="a0_hvg_checkpoint_action0_survival_tradeoff_gate",
)

fig_sparse16_tradeoff_flat, sparse16_tradeoff_flat_rows, sparse16_tradeoff_flat_summary, sparse16_tradeoff_flat_missing = _plot_checkpoint_action0_survival_tradeoff(
    sparse16_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=["flat p0.000", "flat p0.001", "flat p0.003", "flat p0.010"],
    title="A0 Sparse16 flat: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_sparse16_checkpoint_action0_survival_tradeoff_flat",
)

fig_sparse16_tradeoff_gated, sparse16_tradeoff_gated_rows, sparse16_tradeoff_gated_summary, sparse16_tradeoff_gated_missing = _plot_checkpoint_action0_survival_tradeoff(
    sparse16_checkpoint_action0_by_seed,
    plot_group="gated",
    labels=["gated p0.000", "gated p0.001", "gated p0.003", "gated p0.010"],
    title="A0 Sparse16 gated: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_sparse16_checkpoint_action0_survival_tradeoff_gated",
)

fig_aib_tradeoff_flat, aib_tradeoff_flat_rows, aib_tradeoff_flat_summary, aib_tradeoff_flat_missing = _plot_checkpoint_action0_survival_tradeoff(
    aib_checkpoint_action0_by_seed,
    plot_group="flat",
    labels=["baseline", "flat local t0.10", "flat local t0.20", "flat local t0.35", "flat non-idle t0.20"],
    title="A0 AIB flat budget variants: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_aib_checkpoint_action0_survival_tradeoff_flat",
)

fig_aib_tradeoff_gate, aib_tradeoff_gate_rows, aib_tradeoff_gate_summary, aib_tradeoff_gate_missing = _plot_checkpoint_action0_survival_tradeoff(
    aib_checkpoint_action0_by_seed,
    plot_group="gate",
    labels=["baseline", "flat local t0.20", "gate h-greedy t0.20"],
    title="A0 AIB gated budget variant: checkpoint-aligned action-0 / survival tradeoff",
    save_name="a0_aib_checkpoint_action0_survival_tradeoff_gate",
)

COMBINED_TRADEOFF_SOURCES = [
    {
        "tradeoff_family": "HVG heuristic",
        "rows": hvg_tradeoff_heuristic_rows,
        "summary": hvg_tradeoff_heuristic_summary,
    },
    {
        "tradeoff_family": "HVG gate",
        "rows": hvg_tradeoff_gate_rows,
        "summary": hvg_tradeoff_gate_summary,
    },
    {
        "tradeoff_family": "Sparse16 flat",
        "rows": sparse16_tradeoff_flat_rows,
        "summary": sparse16_tradeoff_flat_summary,
    },
    {
        "tradeoff_family": "Sparse16 gated",
        "rows": sparse16_tradeoff_gated_rows,
        "summary": sparse16_tradeoff_gated_summary,
    },
    {
        "tradeoff_family": "AIB flat",
        "rows": aib_tradeoff_flat_rows,
        "summary": aib_tradeoff_flat_summary,
    },
    {
        "tradeoff_family": "AIB gate",
        "rows": aib_tradeoff_gate_rows,
        "summary": aib_tradeoff_gate_summary,
    },
]

combined_seed_frames = []
combined_mean_frames = []
for source in COMBINED_TRADEOFF_SOURCES:
    rows = source["rows"]
    if isinstance(rows, pd.DataFrame) and not rows.empty:
        frame = rows.copy()
        frame["tradeoff_family"] = source["tradeoff_family"]
        frame["variation"] = frame["condition_label"].astype(str)
        combined_seed_frames.append(frame)

    summary = source["summary"]
    if isinstance(summary, pd.DataFrame) and not summary.empty:
        frame = summary.copy()
        frame["tradeoff_family"] = source["tradeoff_family"]
        frame["variation"] = frame["condition_label"].astype(str)
        combined_mean_frames.append(frame)

combined_tradeoff_seed_rows = (
    pd.concat(combined_seed_frames, ignore_index=True, sort=False)
    if combined_seed_frames
    else pd.DataFrame()
)
combined_tradeoff_mean_rows = (
    pd.concat(combined_mean_frames, ignore_index=True, sort=False)
    if combined_mean_frames
    else pd.DataFrame()
)

if combined_tradeoff_seed_rows.empty or combined_tradeoff_mean_rows.empty:
    print("No combined checkpoint-aligned action-0 / survival tradeoff data available.")
else:
    family_order = [source["tradeoff_family"] for source in COMBINED_TRADEOFF_SOURCES]
    family_symbols = {
        "HVG heuristic": "circle",
        "HVG gate": "diamond",
        "Sparse16 flat": "square",
        "Sparse16 gated": "cross",
        "AIB flat": "triangle-up",
        "AIB gate": "x",
    }
    variation_order = list(dict.fromkeys(combined_tradeoff_mean_rows["variation"].astype(str).tolist()))
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24 + px.colors.qualitative.Safe
    variation_colors = {
        variation: A0_HVG_VARIANT_COLORS.get(variation, palette[idx % len(palette)])
        for idx, variation in enumerate(variation_order)
    }

    combined_tradeoff_audit = combined_tradeoff_mean_rows.copy().rename(columns={
        "mean_action0_fraction": "condition_mean_plotted_action0_fraction",
        "std_action0_fraction": "condition_std_plotted_action0_fraction",
        "mean_survival_pct": "condition_mean_plotted_survival_pct",
        "std_survival_pct": "condition_std_plotted_survival_pct",
    })
    combined_tradeoff_audit_columns = [
        "tradeoff_family",
        "variation",
        "condition_mean_plotted_action0_fraction",
        "condition_std_plotted_action0_fraction",
        "condition_mean_plotted_survival_pct",
        "condition_std_plotted_survival_pct",
        "n_seeds",
        "seeds",
        "runs",
        "checkpoint_files",
        "survival_metrics",
    ]
    combined_tradeoff_audit_columns = [
        column for column in combined_tradeoff_audit_columns
        if column in combined_tradeoff_audit.columns
    ]
    combined_tradeoff_audit = (
        combined_tradeoff_audit[combined_tradeoff_audit_columns]
        .sort_values(["tradeoff_family", "variation"], kind="stable")
        .reset_index(drop=True)
    )
    if SHOW_CHECKPOINT_AUDIT_TABLES:
        print("Checkpoint tradeoff average values used for: all A0 families combined")
        display(combined_tradeoff_audit)

    fig_combined_checkpoint_action0_survival_tradeoff = go.Figure()
    for family in family_order:
        for variation in variation_order:
            mean_point = combined_tradeoff_mean_rows[
                combined_tradeoff_mean_rows["tradeoff_family"].astype(str).eq(family)
                & combined_tradeoff_mean_rows["variation"].astype(str).eq(variation)
            ]
            if mean_point.empty:
                continue
            mean_row = mean_point.iloc[0]
            color = variation_colors[variation]
            symbol = family_symbols[family]
            fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
                go.Scatter(
                    x=[mean_row["mean_action0_fraction"]],
                    y=[mean_row["mean_survival_pct"]],
                    mode="markers",
                    name=f"{family} | {variation}",
                    legendgroup=f"data:{family}:{variation}",
                    showlegend=False,
                    marker={
                        "color": color,
                        "symbol": symbol,
                        "size": 10,
                        "opacity": 0.95,
                        "line": {"color": "black", "width": 1.1},
                    },
                    customdata=np.array([[
                        family,
                        variation,
                        mean_row["n_seeds"],
                        str(mean_row["seeds"]),
                        str(mean_row["runs"]),
                        str(mean_row["checkpoint_files"]),
                        str(mean_row["survival_metrics"]),
                        mean_row["std_action0_fraction"],
                        mean_row["std_survival_pct"],
                    ]], dtype=object),
                    hovertemplate=(
                        "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
                        "mean action-0=%{x:.4f}<br>"
                        "std action-0=%{customdata[7]:.4f}<br>"
                        "mean survival=%{y:.2f}%<br>"
                        "std survival=%{customdata[8]:.2f}%<br>"
                        "n_seeds=%{customdata[2]}<br>"
                        "seeds=%{customdata[3]}<br>"
                        "runs=%{customdata[4]}<br>"
                        "checkpoints=%{customdata[5]}<br>"
                        "survival metrics=%{customdata[6]}<extra></extra>"
                    ),
                )
            )

    # Compact legend keys: shape encodes the broader run family; color encodes the variation.
    family_legend_labels = {
        "HVG heuristic": "HVG heur.",
        "HVG gate": "HVG gate",
        "Sparse16 flat": "Sparse flat",
        "Sparse16 gated": "Sparse gated",
        "AIB flat": "AIB flat",
        "AIB gate": "AIB gate",
    }
    for family in family_order:
        fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=family_legend_labels.get(family, family),
                legendgroup="family_shape",
                legendgrouptitle_text="shape = family",
                marker={
                    "symbol": family_symbols[family],
                    "color": "#4a4a4a",
                    "size": 8,
                    "line": {"color": "black", "width": 0.7},
                },
                hoverinfo="skip",
            )
        )
    for variation in variation_order:
        fig_combined_checkpoint_action0_survival_tradeoff.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=str(variation),
                legendgroup="variation_color",
                legendgrouptitle_text="color = variation",
                marker={
                    "symbol": "circle",
                    "color": variation_colors[variation],
                    "size": 8,
                    "line": {"color": "black", "width": 0.4},
                },
                hoverinfo="skip",
            )
        )

    fig_combined_checkpoint_action0_survival_tradeoff.add_annotation(
        text="one point = mean over seeds",
        x=0.01,
        y=102,
        xref="x",
        yref="y",
        showarrow=False,
        font={"size": 11, "color": "#333"},
        align="left",
    )
    fig_combined_checkpoint_action0_survival_tradeoff.add_annotation(
        text="better",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-55,
        ay=35,
        font={"size": 11, "color": "#333"},
        arrowcolor="#333",
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_layout(
        title=(
            "All A0 families: mean checkpoint-aligned action-0 / survival tradeoff"
            "<br><sup>shape = run family; color = variation; one point per condition mean</sup>"
        ),
        template="plotly_white",
        height=650,
        width=1180,
        hovermode="closest",
        legend={
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 10},
            "itemsizing": "constant",
        },
        margin={"l": 75, "r": 280, "t": 105, "b": 70},
    )
    combined_x_range = _plot_range_with_padding(
        combined_tradeoff_mean_rows["mean_action0_fraction"],
        lower=0,
        upper=1,
        min_span=0.08,
    )
    combined_y_range = _plot_range_with_padding(
        combined_tradeoff_mean_rows["mean_survival_pct"],
        lower=0,
        upper=105,
        min_span=8,
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_xaxes(
        title_text="mean action-0 fraction across agents",
        range=combined_x_range,
    )
    fig_combined_checkpoint_action0_survival_tradeoff.update_yaxes(
        title_text="episodic survival (%)",
        range=combined_y_range,
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_all_families_checkpoint_action0_survival_tradeoff.html"
        fig_combined_checkpoint_action0_survival_tradeoff.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_combined_checkpoint_action0_survival_tradeoff.show()


print(
    "Teacher-student BC action-0 / survival tradeoff skipped: "
    "the BC epoch metrics contain action imitation metrics but no episodic-survival rollout metric aligned to those offline checkpoints."
)


## Pareto Frontier: Action-0 / Survival Tradeoff

This plot keeps one averaged point per condition and highlights the Pareto frontier for the two objectives we want to maximize: action-0 usage and episodic survival. A condition is Pareto-optimal if no other condition has both higher or equal action-0 usage and higher or equal survival, with at least one strict improvement.


In [ ]:
def _mark_pareto_frontier(data, *, x_col, y_col):
    if data.empty:
        out = data.copy()
        out["is_pareto"] = pd.Series(dtype=bool)
        return out

    out = data.copy()
    out[x_col] = pd.to_numeric(out[x_col], errors="coerce")
    out[y_col] = pd.to_numeric(out[y_col], errors="coerce")
    values = out[[x_col, y_col]].to_numpy(dtype=float)
    is_pareto = []
    for idx, (x_value, y_value) in enumerate(values):
        if np.isnan(x_value) or np.isnan(y_value):
            is_pareto.append(False)
            continue
        dominated = np.any(
            (values[:, 0] >= x_value)
            & (values[:, 1] >= y_value)
            & ((values[:, 0] > x_value) | (values[:, 1] > y_value))
        )
        is_pareto.append(not bool(dominated))
    out["is_pareto"] = is_pareto
    return out


if "combined_tradeoff_mean_rows" not in globals() or combined_tradeoff_mean_rows.empty:
    print("Run the checkpoint-aligned tradeoff cell above before computing the Pareto frontier.")
    pareto_tradeoff_rows = pd.DataFrame()
    pareto_frontier_rows = pd.DataFrame()
else:
    pareto_tradeoff_rows = combined_tradeoff_mean_rows.copy()
    pareto_tradeoff_rows["variation"] = pareto_tradeoff_rows["variation"].astype(str)

    # The combined comparison table intentionally reuses comparator runs, such as
    # the baseline or flat local t0.20, in several panels. For the Pareto frontier,
    # keep each unique run variant once so duplicated panel membership cannot
    # create duplicated frontier points.
    pareto_tradeoff_rows["_runs_key"] = pareto_tradeoff_rows["runs"].map(str)
    pareto_tradeoff_rows["_checkpoint_key"] = pareto_tradeoff_rows["checkpoint_files"].map(str)
    pareto_tradeoff_rows = (
        pareto_tradeoff_rows
        .drop_duplicates(
            [
                "variation",
                "_runs_key",
                "_checkpoint_key",
                "mean_action0_fraction",
                "mean_survival_pct",
            ],
            keep="first",
        )
        .drop(columns=["_runs_key", "_checkpoint_key"])
        .reset_index(drop=True)
    )

    pareto_tradeoff_rows = _mark_pareto_frontier(
        pareto_tradeoff_rows,
        x_col="mean_action0_fraction",
        y_col="mean_survival_pct",
    )
    pareto_frontier_rows = (
        pareto_tradeoff_rows[pareto_tradeoff_rows["is_pareto"]]
        .sort_values(["mean_action0_fraction", "mean_survival_pct"], kind="stable")
        .reset_index(drop=True)
    )

    pareto_frontier_table = pareto_frontier_rows[[
        "tradeoff_family",
        "variation",
        "mean_action0_fraction",
        "std_action0_fraction",
        "mean_survival_pct",
        "std_survival_pct",
        "n_seeds",
        "seeds",
        "runs",
        "checkpoint_files",
        "survival_metrics",
    ]].rename(columns={
        "mean_action0_fraction": "pareto_mean_action0_fraction",
        "std_action0_fraction": "pareto_std_action0_fraction",
        "mean_survival_pct": "pareto_mean_survival_pct",
        "std_survival_pct": "pareto_std_survival_pct",
    })
    print(f"Pareto frontier conditions: {len(pareto_frontier_table)} / {len(pareto_tradeoff_rows)}")
    display(pareto_frontier_table)

    fig_pareto_checkpoint_action0_survival = go.Figure()
    for family in family_order:
        for variation in variation_order:
            points = pareto_tradeoff_rows[
                pareto_tradeoff_rows["tradeoff_family"].astype(str).eq(family)
                & pareto_tradeoff_rows["variation"].astype(str).eq(variation)
            ]
            if points.empty:
                continue
            row = points.iloc[0]
            is_pareto = bool(row["is_pareto"])
            color = variation_colors.get(variation, "#999999")
            symbol = family_symbols.get(family, "circle")
            fig_pareto_checkpoint_action0_survival.add_trace(
                go.Scatter(
                    x=[row["mean_action0_fraction"]],
                    y=[row["mean_survival_pct"]],
                    mode="markers",
                    name=f"{family} | {variation}",
                    showlegend=False,
                    marker={
                        "color": color,
                        "symbol": symbol,
                        "size": 11 if is_pareto else 6,
                        "opacity": 0.96 if is_pareto else 0.20,
                        "line": {
                            "color": "black" if is_pareto else "rgba(80,80,80,0.35)",
                            "width": 1.5 if is_pareto else 0.5,
                        },
                    },
                    customdata=np.array([[
                        family,
                        variation,
                        "Pareto" if is_pareto else "dominated",
                        row["n_seeds"],
                        str(row["seeds"]),
                        str(row["runs"]),
                        str(row["checkpoint_files"]),
                        str(row["survival_metrics"]),
                        row["std_action0_fraction"],
                        row["std_survival_pct"],
                    ]], dtype=object),
                    hovertemplate=(
                        "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
                        "status=%{customdata[2]}<br>"
                        "mean action-0=%{x:.4f}<br>"
                        "std action-0=%{customdata[8]:.4f}<br>"
                        "mean survival=%{y:.2f}%<br>"
                        "std survival=%{customdata[9]:.2f}%<br>"
                        "n_seeds=%{customdata[3]}<br>"
                        "seeds=%{customdata[4]}<br>"
                        "runs=%{customdata[5]}<br>"
                        "checkpoints=%{customdata[6]}<br>"
                        "survival metrics=%{customdata[7]}<extra></extra>"
                    ),
                )
            )

    frontier_line = pareto_frontier_rows.drop_duplicates([
        "mean_action0_fraction",
        "mean_survival_pct",
    ]).sort_values(["mean_action0_fraction", "mean_survival_pct"])
    if len(frontier_line) >= 2:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=frontier_line["mean_action0_fraction"],
                y=frontier_line["mean_survival_pct"],
                mode="lines",
                name="Pareto frontier",
                showlegend=True,
                line={"color": "black", "width": 2.2, "dash": "dash"},
                hoverinfo="skip",
            )
        )

    # Compact legend keys.
    family_legend_labels = {
        "HVG heuristic": "HVG heur.",
        "HVG gate": "HVG gate",
        "Sparse16 flat": "Sparse flat",
        "Sparse16 gated": "Sparse gated",
        "AIB flat": "AIB flat",
        "AIB gate": "AIB gate",
    }
    for family in family_order:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=family_legend_labels.get(family, family),
                legendgroup="family_shape",
                legendgrouptitle_text="shape = family",
                marker={
                    "symbol": family_symbols.get(family, "circle"),
                    "color": "#4a4a4a",
                    "size": 8,
                    "line": {"color": "black", "width": 0.7},
                },
                hoverinfo="skip",
            )
        )
    for variation in variation_order:
        fig_pareto_checkpoint_action0_survival.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=str(variation),
                legendgroup="variation_color",
                legendgrouptitle_text="color = variation",
                marker={
                    "symbol": "circle",
                    "color": variation_colors.get(variation, "#999999"),
                    "size": 8,
                    "line": {"color": "black", "width": 0.4},
                },
                hoverinfo="skip",
            )
        )
    fig_pareto_checkpoint_action0_survival.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name="Pareto point",
            legendgroup="pareto_status",
            legendgrouptitle_text="frontier",
            marker={
                "symbol": "circle",
                "color": "white",
                "size": 10,
                "line": {"color": "black", "width": 1.5},
            },
            hoverinfo="skip",
        )
    )
    fig_pareto_checkpoint_action0_survival.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name="dominated",
            legendgroup="pareto_status",
            marker={
                "symbol": "circle",
                "color": "rgba(120,120,120,0.25)",
                "size": 6,
                "line": {"color": "rgba(80,80,80,0.35)", "width": 0.5},
            },
            hoverinfo="skip",
        )
    )

    fig_pareto_checkpoint_action0_survival.add_annotation(
        text="Pareto frontier: maximize survival and action-0",
        x=0.01,
        y=102,
        xref="x",
        yref="y",
        showarrow=False,
        font={"size": 11, "color": "#333"},
        align="left",
    )
    fig_pareto_checkpoint_action0_survival.add_annotation(
        text="better",
        x=0.98,
        y=102,
        xref="x",
        yref="y",
        showarrow=True,
        ax=-55,
        ay=35,
        font={"size": 11, "color": "#333"},
        arrowcolor="#333",
    )
    fig_pareto_checkpoint_action0_survival.update_layout(
        title=(
            "All A0 families: Pareto frontier of mean checkpoint-aligned action-0 / survival"
            "<br><sup>non-dominated points are highlighted; shape = family; color = variation</sup>"
        ),
        template="plotly_white",
        height=650,
        width=1180,
        hovermode="closest",
        legend={
            "orientation": "v",
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.01,
            "font": {"size": 10},
            "itemsizing": "constant",
        },
        margin={"l": 75, "r": 280, "t": 105, "b": 70},
    )
    pareto_x_range = _plot_range_with_padding(
        pareto_tradeoff_rows["mean_action0_fraction"],
        lower=0,
        upper=1,
        min_span=0.08,
    )
    pareto_y_range = _plot_range_with_padding(
        pareto_tradeoff_rows["mean_survival_pct"],
        lower=0,
        upper=105,
        min_span=8,
    )
    fig_pareto_checkpoint_action0_survival.update_xaxes(
        title_text="mean action-0 fraction across agents",
        range=pareto_x_range,
    )
    fig_pareto_checkpoint_action0_survival.update_yaxes(
        title_text="episodic survival (%)",
        range=pareto_y_range,
    )

    if SAVE_FIGURES:
        save_path = ctx["FIG_DIR"] / "a0_all_families_checkpoint_action0_survival_pareto_frontier.html"
        fig_pareto_checkpoint_action0_survival.write_html(save_path, include_plotlyjs="cdn")
        print(f"Saved: {save_path}")
    if SHOW_FIGURES:
        fig_pareto_checkpoint_action0_survival.show()
